# Studio qualità — Extract (NER, Wikidata, Geocoding)

Analisi **as-is** di `pathos extract` (NER `xx_ent_wiki_sm`, geocoding Nominatim, linking Wikidata)
sul DB reale. Nessun fix qui: solo mappatura di copertura, rumore, criticità con esempi concreti.

Riferimento codice: `pathosphere/semantic/extract.py`.

Nota di stato: il fix stoplist/rate-limit Wikidata (`fix/wikidata-linking`) è stato mergiato su
`main` prima di questo notebook, ma il lookup budget (`--max-lookups`, rate limit Wikimedia)
resta stretto: la copertura QID sotto riflette quanto già processato con budget limitato a
50 lookups/run, non un backfill completo.

In [1]:
import sys, struct, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pathosphere").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from pathosphere.db.schema import get_connection

DB_PATH = REPO_ROOT / "data" / "db" / "pathosphere.db"
conn = get_connection(DB_PATH)
print(f"DB: {DB_PATH}  exists={DB_PATH.exists()}")

def q(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

q("SELECT COUNT(*) AS raw_documents FROM raw_documents")


DB: /Users/dom/Documents/GitHub/pathosphere/data/db/pathosphere.db  exists=True


,raw_documents
0,176477


## 1. Copertura NER / entità

In [2]:
ent_by_type = q("SELECT entity_type, COUNT(*) AS n FROM entities GROUP BY entity_type ORDER BY n DESC")
ent_by_type

,entity_type,n
0,other,4439
1,company,3254
2,location,2035
3,person,1739


In [3]:
docs_ner = q("SELECT SUM(ner_done) AS done, COUNT(*) AS total FROM raw_documents")
docs_ner

,done,total
0,129872,176477


## 2. Rapporto segnale/rumore — top entità per mentions

In [4]:
top_mentions = q("""
SELECT e.name, e.entity_type, SUM(de.mentions) AS total_mentions, COUNT(DISTINCT de.document_id) AS n_docs
FROM document_entities de JOIN entities e ON e.id = de.entity_id
GROUP BY e.id ORDER BY total_mentions DESC LIMIT 40
""")
top_mentions

,name,entity_type,total_mentions,n_docs
0,GDELT,company,128082,128082
1,POLICE,company,10443,10407
2,GOVERNMENT,other,5148,5126
3,IRAN,location,3843,3792
4,SCHOOL,other,3084,3032
5,PRISON,company,2565,2560
6,MEDIA,other,2454,2443
7,STUDENT,other,2266,2255
8,US,location,2083,1930
9,PRESIDENT,other,2078,2077


In [5]:
generic_lookalike = top_mentions[
    top_mentions["name"].str.isupper() & (top_mentions["name"].str.split().str.len() == 1)
]
print(f"Tra le top 40 entità per mentions, {len(generic_lookalike)} sono mono-parola ALL CAPS "
      "(potenzialmente generiche/rumore NER su testo GDELT).")
generic_lookalike[["name", "entity_type", "total_mentions"]]

Tra le top 40 entità per mentions, 37 sono mono-parola ALL CAPS (potenzialmente generiche/rumore NER su testo GDELT).


,name,entity_type,total_mentions
0,GDELT,company,128082
1,POLICE,company,10443
2,GOVERNMENT,other,5148
3,IRAN,location,3843
4,SCHOOL,other,3084
5,PRISON,company,2565
6,MEDIA,other,2454
7,STUDENT,other,2266
8,US,location,2083
9,PRESIDENT,other,2078


### 2b. "GDELT" come entità — leak del prefisso titolo nel NER

La entità col maggior numero di mention in assoluto è letteralmente `GDELT`. I titoli
sintetici generati dall'ingestor GDELT hanno formato `"GDELT: ACTOR → ACTOR2"`
(vedi es. sopra): la parola `GDELT` viene quindi taggata come ORG/company dal NER su
(quasi) ogni documento di origine GDELT.

In [6]:
gdelt_ent = q("""
SELECT e.id, SUM(de.mentions) AS mentions, COUNT(DISTINCT de.document_id) AS n_docs
FROM entities e JOIN document_entities de ON de.entity_id = e.id
WHERE e.name = 'GDELT'
GROUP BY e.id
""")
gdelt_docs_total = q("SELECT COUNT(*) AS n FROM raw_documents WHERE origin = 'gdelt'")
n_gdelt_docs = int(gdelt_docs_total["n"][0])
n_ent_docs = int(gdelt_ent["n_docs"][0]) if len(gdelt_ent) else 0
print(f"Entità 'GDELT': presente in {n_ent_docs} documenti su {n_gdelt_docs} raw_documents "
      f"origin=gdelt ({n_ent_docs / n_gdelt_docs * 100:.1f}% se n_gdelt_docs>0).")
print("Causa: il prefisso letterale del titolo sintetico finisce nel testo passato al NER "
      "(_build_text in semantic/extract.py concatena title+body senza rimuovere il prefisso).")

Entità 'GDELT': presente in 128082 documenti su 174286 raw_documents origin=gdelt (73.5% se n_gdelt_docs>0).
Causa: il prefisso letterale del titolo sintetico finisce nel testo passato al NER (_build_text in semantic/extract.py concatena title+body senza rimuovere il prefisso).


### 2c. Frammenti HTML come entità

Controlliamo se tag/frammenti HTML residui nel body (es. `<span>`, `<strong>`, `</p>`)
vengono taggati come entità — indicherebbe che il body non è ripulito da markup prima del NER.

In [7]:
html_like = q("""
SELECT e.name, e.entity_type, SUM(de.mentions) AS mentions
FROM entities e JOIN document_entities de ON de.entity_id = e.id
WHERE e.name LIKE '%<%' OR e.name LIKE '%>%' OR e.name LIKE '%</%'
GROUP BY e.id ORDER BY mentions DESC LIMIT 20
""")
print(f"Entità con frammenti '<' o '>' nel nome: {len(html_like)}")
html_like

Entità con frammenti '<' o '>' nel nome: 20


,name,entity_type,mentions
0,span><strong,other,48
1,said.</p,person,30
2,Times Nigeria</a>.</p,other,30
3,figcaption><a,location,13
4,Sunday.</p,other,11
5,added.</p,other,11
6,"figcaption><a href=""https://www.rt.com",company,7
7,The Indian Express</em,other,7
8,"figcaption><a href=""https://www.rt.com/",company,6
9,country.</p,other,6


## 3. Copertura Wikidata QID

In [8]:
qid_cov = q("""
SELECT
  COUNT(*) AS total,
  SUM(wikidata_checked) AS checked,
  SUM(CASE WHEN wikidata_qid IS NOT NULL THEN 1 ELSE 0 END) AS has_qid
FROM entities
""")
qid_cov

,total,checked,has_qid
0,11467,191,34


In [9]:
unchecked_top = q("""
SELECT e.name, e.entity_type, SUM(de.mentions) AS mentions
FROM entities e JOIN document_entities de ON de.entity_id = e.id
WHERE e.wikidata_checked = 0
GROUP BY e.id ORDER BY mentions DESC LIMIT 20
""")
unchecked_top

,name,entity_type,mentions
0,MINNEAPOLIS,other,357
1,AUSTRALIAN,other,335
2,FRANCE,company,317
3,HIGH COURT,other,317
4,SOUTH,other,314
5,NATO,company,313
6,NEW YORK,location,311
7,WEBSITE,company,302
8,EMPLOYER,company,301
9,EU,company,301


## 4. NER signal-to-noise — esempi testo reale vs entità estratte

In [10]:
random.seed(42)
sample_docs = q("SELECT id, title, body FROM raw_documents WHERE ner_done = 1 ORDER BY RANDOM() LIMIT 6")
for _, d in sample_docs.iterrows():
    ents = q(
        "SELECT e.name, e.entity_type FROM document_entities de JOIN entities e ON e.id = de.entity_id "
        "WHERE de.document_id = ?",
        params=(int(d.id),),
    )
    print(f"\n--- doc {d.id} ---")
    print((d.title or "")[:150])
    print((d.body or "")[:300])
    print("entità estratte:", list(ents.itertuples(index=False, name=None)))


--- doc 152633 ---
GDELT: ADVOCATE →  [100]

entità estratte: [('GDELT', 'company'), ('ADVOCATE', 'other')]

--- doc 41151 ---
GDELT: GOVERNOR →  [110]

entità estratte: [('GDELT', 'company'), ('GOVERNOR', 'other')]

--- doc 138560 ---
GDELT: VILLAGE →  [150]

entità estratte: [('GDELT', 'company')]

--- doc 31185 ---
GDELT: DISTRICT COURT → SECRETARY OF STATE [110]

entità estratte: [('GDELT', 'company'), ('COURT', 'company')]

--- doc 62084 ---
GDELT: GOVERNMENT → UNITED KINGDOM [173]

entità estratte: [('GDELT', 'company'), ('UNITED KINGDOM', 'other'), ('GOVERNMENT', 'other')]

--- doc 148746 ---
GDELT:  → POLICE [1823]

entità estratte: [('GDELT', 'company'), ('POLICE', 'company')]


## 5. Copertura geocoding

In [11]:
geo_cov = q("""
SELECT COUNT(*) AS events_with_location,
       SUM(CASE WHEN lat IS NOT NULL THEN 1 ELSE 0 END) AS geocoded
FROM events WHERE location_name IS NOT NULL
""")
geo_cov

,events_with_location,geocoded
0,109701,108970


In [12]:
cache = q("""
SELECT COUNT(*) AS total, SUM(CASE WHEN lat IS NULL THEN 1 ELSE 0 END) AS misses
FROM geocode_cache
""")
cache

,total,misses
0,28,4


In [13]:
misses_sample = q("SELECT query FROM geocode_cache WHERE lat IS NULL LIMIT 30")
misses_sample

,query
0,Oresund Strait
1,Luzon Strait
2,Ombai Strait
3,Bohai Strait


## Sintesi criticità osservate in questo run

In [14]:
total_ent = int(qid_cov["total"][0])
has_qid = int(qid_cov["has_qid"][0])
checked = int(qid_cov["checked"][0])
misses = int(cache["misses"][0]) if cache["misses"][0] is not None else 0
cache_total = int(cache["total"][0])

print("--- Sintesi (valori da questo run) ---")
print(f"1. 'GDELT' è l'entità con più mention in assoluto: {n_ent_docs} documenti "
      f"({n_ent_docs / n_gdelt_docs * 100:.1f}% dei doc origin=gdelt se n_gdelt_docs>0) — "
      "leak del prefisso titolo sintetico nel NER, non un'entità reale.")
print(f"2. Frammenti HTML taggati come entità: {len(html_like)} — il body non è ripulito "
      "da markup prima del NER.")
print(f"3. Entità totali: {total_ent} | con QID: {has_qid} ({has_qid/total_ent*100:.1f}%) | "
      f"checked: {checked} ({checked/total_ent*100:.1f}%) — budget lookup/notte è il collo di bottiglia.")
print(f"4. Entità generiche ALL CAPS ancora tra le top-40 per mentions: {len(generic_lookalike)}.")
print(f"5. Geocode cache: {misses}/{cache_total} query MAI risolte "
      f"({misses/cache_total*100:.1f}% se cache_total>0) — cache misses non hanno scadenza, "
      "restano bloccate per sempre salvo retry manuale.")

--- Sintesi (valori da questo run) ---
1. 'GDELT' è l'entità con più mention in assoluto: 128082 documenti (73.5% dei doc origin=gdelt se n_gdelt_docs>0) — leak del prefisso titolo sintetico nel NER, non un'entità reale.
2. Frammenti HTML taggati come entità: 20 — il body non è ripulito da markup prima del NER.
3. Entità totali: 11467 | con QID: 34 (0.3%) | checked: 191 (1.7%) — budget lookup/notte è il collo di bottiglia.
4. Entità generiche ALL CAPS ancora tra le top-40 per mentions: 37.
5. Geocode cache: 4/28 query MAI risolte (14.3% se cache_total>0) — cache misses non hanno scadenza, restano bloccate per sempre salvo retry manuale.
